**ПЕРВОЕ ЗАДАНИЕ**

После изучения таблицы стало понятно, что из-за разных event_label при одинковых event_action и городе, одной и той же паре (event_action, город) соответствует сразу несколько строк таблицы, поэтому было решено использовать строку с максимальным значением
unique_events, так как более вероятно, что именно такой event_label получало большинство пользователей

1. Сколько людей пользуются фильтрами?
В качестве оценки числа уникальных пользователей в силу отсутствия точной метрики по их количеству я решил приблизительно считать "1 сессия ~ 1 человек", будем рассматривать событие "search-tools-button_open"

In [10]:
import pandas as pd
ed = pd.read_csv('./1_task_table.csv')
ed['event_label'] = ed['event_label'].str.split(' / ').str[0].str.strip()

2. В каких городах фильтрами пользуются больше? В каких меньше?
Построим отдельный DataFrame, где будут число уникальных визитов, число пользователей, использующих фильтрацию, а так доля пользователей, использующих фильтры, 
и, к примеру, возьмем 10 городов с начала и 10 с конца

In [11]:
cleaned_ed = ed.groupby(['event_label', 'event_action']).agg({
    'unique_events': 'max'
}).reset_index().dropna(axis=0, how='all')
# Сколько людей пользуются фильтрами, будем брать максимум, так как это наиболее вероятный вариант event_label, который показыается всем или почти все пользователям
total_visits = cleaned_ed[cleaned_ed['event_action'] == 'Page Visit']['unique_events'].sum()
print(f"Общее число уникальных визитов: {total_visits}")
total_filters = cleaned_ed[cleaned_ed['event_action'] == 'search-tools-button_open']['unique_events'].sum()
print(f"Общее число пользователей, использующих фильтры: {total_filters}")
print(f"Процент пользователей, использующих фильтры: {total_filters / total_visits * 100:.2g}%")

Общее число уникальных визитов: 1384468.0
Общее число пользователей, использующих фильтры: 18253.0
Процент пользователей, использующих фильтры: 1.3%


In [12]:
visits = cleaned_ed[cleaned_ed['event_action'] == 'Page Visit'].set_index('event_label')['unique_events']
filters = cleaned_ed[cleaned_ed['event_action'] == 'search-tools-button_open'].set_index('event_label')['unique_events']

city_stats = pd.DataFrame({'visits': visits, 'filters': filters}).dropna(axis=0, how='any')
city_stats['filter_rate(%)'] = city_stats['filters'] / city_stats['visits'] * 100
city_stats.sort_values('filter_rate(%)', ascending=True, inplace=True)

print('\nГорода с наименьшим числом пользователей, использующих фильтры')
print(city_stats.iloc[:10])
print('\nГорода с наименьшим числом пользователей, использующих фильтры')
print(city_stats.iloc[-10:]) 



Города с наименьшим числом пользователей, использующих фильтры
             visits  filters  filter_rate(%)
event_label                                 
New Athos    2155.0      5.0        0.232019
Nha Trang     378.0      1.0        0.264550
Mishor        363.0      1.0        0.275482
Chernobyl    3334.0     10.0        0.299940
Goa           920.0      3.0        0.326087
Abrau-Durso  4148.0     15.0        0.361620
Klin          274.0      1.0        0.364964
Polotsk       248.0      1.0        0.403226
Maykop       1601.0      7.0        0.437227
Izmir         224.0      1.0        0.446429

Города с наименьшим числом пользователей, использующих фильтры
                     visits  filters  filter_rate(%)
event_label                                         
Novokuznetsk            6.0      1.0       16.666667
Amiens                  5.0      1.0       20.000000
bogolyubovo             5.0      1.0       20.000000
Vicenza                 5.0      1.0       20.000000
Partenit      

3. Какие разделы фильтров наиболее востребованы? “фильтры”, “сортировка”, “категории”?
Для начала изменим event_action на более понятные и сгруппированные по смыслу названия, затем посчитаем уникальные использования для каждой секции и выведем процент для каждой от общего числа использований фильтров

In [13]:
#Создаём понятные разделы
def get_filter_section(action):
    if action == 'filters-categories_click':
        return 'Категории + Сортировка'
    elif action == 'price_first' or action == 'price_second' or action == 'price_third':
        return 'Фильтры по цене'
    elif action == 'start_date_click' or action == 'dates_filter_mobile'  or action == 'end_date_click':
        return 'Фильтры по датам'
    elif action == 'ticket-type_checkbox':
        return 'Тип билета'
    elif action == 'pay-type_checkbox':
        return 'Тип оплаты'

ed['section'] = ed['event_action'].apply(get_filter_section)

section_stats = ed.groupby([
    'event_label','section'
]).agg({
    'unique_events': 'max'
}).groupby([
    'section'
]).agg({
    'unique_events': 'sum'
})

# Форматируем вывод
section_stats['% от всех фильтров'] = (
    section_stats['unique_events'] / section_stats['unique_events'].sum() * 100
).round(1)

print(section_stats)

                        unique_events  % от всех фильтров
section                                                  
Категории + Сортировка        10188.0                21.8
Тип билета                     9889.0                21.1
Тип оплаты                     5459.0                11.7
Фильтры по датам              14538.0                31.1
Фильтры по цене                6684.0                14.3


4. Как часто люди пользуются выбором цены?

In [14]:
price_events = ['price_first', 'price_second', 'price_third', 'price_button_submit']
price_usage = cleaned_ed[cleaned_ed['event_action'].isin(price_events)]['unique_events'].sum()
print(f"Уникальных сессий с выбором цены: {price_usage}")
print(f"% от посетителей: {price_usage / total_visits * 100:.2g}%")

Уникальных сессий с выбором цены: 15002.0
% от посетителей: 1.1%


**ВТОРОЕ ЗАДАНИЕ**

Здесь приведен фрагмент кода, который расчитывает p-value и z-статистику нашего теста

In [ ]:
from statsmodels.stats.proportion import proportions_ztest

n_A = 5000
success_A = 300
n_B = 5000
success_B = 450

# Считаем z_stat и p_value через z_test для пропорций
z_stat, p_value = proportions_ztest(
    count=[success_A, success_B],
    nobs=[n_A, n_B],
    alternative='larger'
)
print(f"Z-статистика: {z_stat:.4f}")
print(f"P-value:      {p_value:.10f}  ({p_value:.2e})")
conv_A = success_A / n_A * 100
conv_B = success_B / n_B * 100
lift = (conv_B - conv_A) / conv_A * 100

print(f"\nКонверсия A: {conv_A:.2f}%")
print(f"Конверсия B: {conv_B:.2f}%")
print(f"Относительный рост: +{lift:.1f}%")